# Fundraising → Ballot Mentions: District Explorer

## Question

**How strong is the relationship between fundraising and ballot mentions in each Portland City Council district?**

This notebook is a direct extension of the original weekly-slide Notebook 13b.

It keeps the same simple Plotly + OLS approach, but now makes the **District 1–4 views explicit** rather than treating District 4 as the only presentation zoom.

> This notebook is descriptive. It helps identify cases to investigate; it does not explain *why* a candidate over- or under-performed.


## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from IPython.display import display, clear_output
import ipywidgets as widgets


# ---------------------------------------------------------------
# Find the repository root
# ---------------------------------------------------------------

cwd = Path.cwd().resolve()

for folder in [cwd, *cwd.parents]:
    if (folder / "pyproject.toml").exists():
        ROOT = folder
        break
else:
    raise FileNotFoundError("Could not find the repository root.")


YEAR = 2024

print("ROOT:", ROOT)


## 2. Load the current candidate-level table

We now use the canonical merged table in `data/processed/finance_vs_ballot_support/2024/`.

The original weekly notebook called total private fundraising `fundraising`. The current canonical table calls the same quantity `total_amount`, so we create a simple local alias and leave the analytical logic unchanged.


In [ ]:
data_path = (
    ROOT
    / "data"
    / "processed"
    / "finance_vs_ballot_support"
    / str(YEAR)
    / "candidate_finance_ballot_analysis_2024.csv"
)


analysis = pd.read_csv(data_path)


# Keep the same variable name used in the original weekly notebook.
analysis["fundraising"] = analysis["total_amount"]


plot_data = analysis.dropna(
    subset=[
        "fundraising",
        "mentions",
        "stv_threshold",
    ]
).copy()


print("Candidates with matched fundraising:", len(plot_data))
print()
print("Candidates by district:")

display(
    plot_data["district"]
    .value_counts()
    .sort_index()
    .rename("candidates")
    .to_frame()
)


print()
print("STV threshold values by district:")

display(
    plot_data.groupby("district")["stv_threshold"]
    .agg(["min", "max", "nunique"])
)


## 3. A tiny helper: correlation and R²

For a simple regression with:

`mentions = a + b × fundraising`

the R² is equal to the squared Pearson correlation.


In [ ]:
def relationship_stats(data):
    # Pearson correlation between fundraising and mentions.
    correlation = data["fundraising"].corr(
        data["mentions"]
    )

    # For one-predictor OLS with an intercept:
    r_squared = correlation ** 2

    return correlation, r_squared


## 4. Overall pattern: fundraising and ballot mentions

This keeps the citywide view from the original weekly notebook.


In [ ]:
overall_r, overall_r2 = relationship_stats(
    plot_data
)


fig_overall = px.scatter(
    plot_data,
    x="fundraising",
    y="mentions",
    hover_name="canonical_candidate",
    hover_data={
        "district": True,
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
        "stv_threshold": ":,.0f",
        "is_viable": True,
    },
    trendline="ols",
    template="plotly_white",
    labels={
        "fundraising": "Total fundraising ($)",
        "mentions": "Ballot mentions",
    },
    title=(
        "2024 Portland City Council: "
        "fundraising and ballot mentions"
        f"<br><sup>Pearson r = {overall_r:.2f} | "
        f"R² = {overall_r2:.2f} | "
        f"n = {len(plot_data)}</sup>"
    ),
)


fig_overall.update_traces(
    marker={
        "size": 9,
        "opacity": 0.75,
    }
)


fig_overall.show()


## 5. Compare the four districts

In [ ]:
district_rows = []


for district in sorted(
    plot_data["district"].unique()
):
    district_data = plot_data[
        plot_data["district"].eq(district)
    ]

    r, r2 = relationship_stats(
        district_data
    )

    district_rows.append(
        {
            "district": int(district),
            "candidates": len(district_data),
            "pearson_r": r,
            "r_squared": r2,
        }
    )


district_summary = pd.DataFrame(
    district_rows
)


display(
    district_summary.round(
        {
            "pearson_r": 3,
            "r_squared": 3,
        }
    )
)


In [ ]:
fig_districts = px.bar(
    district_summary,
    x="district",
    y="pearson_r",
    text="pearson_r",
    hover_data={
        "candidates": True,
        "r_squared": ":.3f",
    },
    template="plotly_white",
    labels={
        "district": "District",
        "pearson_r": "Pearson correlation",
    },
    title=(
        "Fundraising → mentions relationship "
        "across districts"
    ),
)


fig_districts.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
)


fig_districts.update_yaxes(
    range=[0, 1]
)


fig_districts.show()


## 6. District-by-district figures

These are the same fundraising → mentions scatter/OLS plots, shown explicitly for Districts 1–4.


In [ ]:
def show_district(selected_district):
    # -----------------------------------------------------------
    # 1. Filter the data
    # -----------------------------------------------------------

    if selected_district == "All":
        data = plot_data.copy()
        title_label = "All districts"

    else:
        data = plot_data[
            plot_data["district"].eq(
                selected_district
            )
        ].copy()

        title_label = (
            f"District {selected_district}"
        )


    # -----------------------------------------------------------
    # 2. Calculate the relationship
    # -----------------------------------------------------------

    r, r2 = relationship_stats(
        data
    )


    # -----------------------------------------------------------
    # 3. Create the Plotly Express scatter
    # -----------------------------------------------------------

    fig = px.scatter(
        data,
        x="fundraising",
        y="mentions",
        hover_name="canonical_candidate",
        hover_data={
            "district": True,
            "fundraising": ":$,.0f",
            "mentions": ":,.0f",
            "first_place_votes": ":,.0f",
            "stv_threshold": ":,.0f",
            "is_viable": True,
        },
        trendline="ols",
        template="plotly_white",
        labels={
            "fundraising": "Total fundraising ($)",
            "mentions": "Ballot mentions",
        },
        title=(
            f"{title_label}: fundraising and ballot mentions"
            f"<br><sup>Pearson r = {r:.2f} | "
            f"R² = {r2:.2f} | "
            f"n = {len(data)}</sup>"
        ),
    )


    fig.update_traces(
        marker={
            "size": 10,
            "opacity": 0.8,
        }
    )


    fig.show()


In [ ]:
for district in [1, 2, 3, 4]:
    show_district(district)


## 7. Optional interactive district explorer

Use the dropdown when you want to move quickly between all districts and one district during exploration.


In [ ]:
district_dropdown = widgets.Dropdown(
    options=[
        ("All districts", "All"),
        ("District 1", 1),
        ("District 2", 2),
        ("District 3", 3),
        ("District 4", 4),
    ],
    value="All",
    description="District:",
)


district_output = widgets.Output()


def update_district(change):
    with district_output:
        clear_output(
            wait=True
        )

        show_district(
            change["new"]
        )


district_dropdown.observe(
    update_district,
    names="value",
)


display(
    district_dropdown,
    district_output,
)


# Show the initial graph.
with district_output:
    show_district(
        district_dropdown.value
    )


## 8. Find candidate pairs with similar fundraising but different support

Same exploratory case-finding logic as Notebook 13b:

- fundraising difference ≤ **10%**
- mentions difference ≥ **50%**

These thresholds are descriptive, not a statistical test.


In [ ]:
MONEY_TOLERANCE = 0.10
MENTION_GAP = 0.50


pair_rows = []


for district in sorted(
    plot_data["district"].unique()
):
    district_data = (
        plot_data[
            plot_data["district"].eq(
                district
            )
        ]
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(district_data)
    ):
        for j in range(
            i + 1,
            len(district_data),
        ):
            candidate_a = (
                district_data.iloc[i]
            )

            candidate_b = (
                district_data.iloc[j]
            )


            # -----------------------------------------------
            # Difference in fundraising
            # -----------------------------------------------

            larger_money = max(
                candidate_a["fundraising"],
                candidate_b["fundraising"],
            )

            money_gap = (
                abs(
                    candidate_a["fundraising"]
                    - candidate_b["fundraising"]
                )
                / larger_money
            )


            # -----------------------------------------------
            # Difference in mentions
            # -----------------------------------------------

            larger_mentions = max(
                candidate_a["mentions"],
                candidate_b["mentions"],
            )

            mention_gap = (
                abs(
                    candidate_a["mentions"]
                    - candidate_b["mentions"]
                )
                / larger_mentions
            )


            # -----------------------------------------------
            # Keep only interesting pairs
            # -----------------------------------------------

            if (
                money_gap <= MONEY_TOLERANCE
                and mention_gap >= MENTION_GAP
            ):
                pair_rows.append(
                    {
                        "district": int(district),
                        "candidate_a": candidate_a[
                            "canonical_candidate"
                        ],
                        "candidate_b": candidate_b[
                            "canonical_candidate"
                        ],
                        "fundraising_a": candidate_a[
                            "fundraising"
                        ],
                        "fundraising_b": candidate_b[
                            "fundraising"
                        ],
                        "mentions_a": candidate_a[
                            "mentions"
                        ],
                        "mentions_b": candidate_b[
                            "mentions"
                        ],
                        "fundraising_gap_pct": (
                            money_gap * 100
                        ),
                        "mentions_gap_pct": (
                            mention_gap * 100
                        ),
                    }
                )


interesting_pairs = pd.DataFrame(
    pair_rows
)


if not interesting_pairs.empty:
    interesting_pairs = (
        interesting_pairs
        .sort_values(
            [
                "district",
                "mentions_gap_pct",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


display(
    interesting_pairs.round(
        {
            "fundraising_a": 0,
            "fundraising_b": 0,
            "mentions_a": 0,
            "mentions_b": 0,
            "fundraising_gap_pct": 1,
            "mentions_gap_pct": 1,
        }
    )
)


## 9. Candidate values by district

In [ ]:
display(
    plot_data[
        [
            "district",
            "canonical_candidate",
            "fundraising",
            "mentions",
            "first_place_votes",
            "stv_threshold",
            "is_viable",
        ]
    ]
    .sort_values(
        [
            "district",
            "mentions",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        {
            "fundraising": 0,
            "mentions": 0,
            "first_place_votes": 0,
            "stv_threshold": 0,
        }
    )
)
